# 🚀 00 — Environment Setup
## Ekegusii-LLM-Translation · Kineses Cloud / Base Jupyter

> **Run this notebook ONCE after cloning in `start.ipynb`.**

This notebook:
1. Sets working directory to project root
2. Fixes NumPy/Pandas ABI compatibility (critical for Kineses conda)
3. Confirms GPU (NVIDIA A100-SXM4-80GB)
4. Verifies Master Corpus & 0% data leakage

---
| Node Spec | Value |
|-----------|-------|
| Python | 3.11.6 (conda-forge) |
| CPU Cores | 22 |
| RAM | 117.9 GB |
| Disk | 967.64 GB |
| GPU | NVIDIA A100-SXM4-80GB (85.1 GB VRAM) |

In [ ]:
# ============================================================
# CELL 1 — Set working directory & sys.path
# ============================================================
import os, sys

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home     = os.path.expanduser('~')
proj_dir = os.path.join(home, 'Ekegusii-LLM-Translation-main')

if not os.path.isdir(proj_dir):
    raise RuntimeError(
        f'Repository not found at: {proj_dir}\n'
        'Run start.ipynb -> Cell 2 (FRESH CLONE) first!'
    )

os.chdir(proj_dir)
if proj_dir not in sys.path:
    sys.path.insert(0, proj_dir)

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')

In [ ]:
# ============================================================
# CELL 2 — Fix NumPy / Pandas ABI Incompatibility
# ============================================================
import subprocess, sys, importlib

def get_version(pkg):
    try:
        m = importlib.import_module(pkg)
        return getattr(m, '__version__', 'unknown')
    except Exception:
        return 'IMPORT_ERROR'

np_ver  = get_version('numpy')
pd_ver  = get_version('pandas')
print(f'numpy  version: {np_ver}')
print(f'pandas version: {pd_ver}')

needs_fix = (
    pd_ver != 'IMPORT_ERROR'
    and pd_ver.startswith('3')
    and np_ver != 'IMPORT_ERROR'
    and np_ver.startswith('1.')
)
if pd_ver == 'IMPORT_ERROR':
    needs_fix = True

if not needs_fix:
    print('No ABI conflict detected. Skipping fix.')
else:
    print('ABI mismatch detected! Attempting fix...')
    print('  [A] Upgrading numpy >= 2.0...', end=' ')
    rA = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy>=2.0.0'],
        capture_output=True, text=True
    )
    if rA.returncode == 0:
        print('OK')
        print('numpy upgraded successfully!')
        print('RESTART KERNEL NOW: Kernel -> Restart Kernel, then re-run from Cell 1')
    else:
        print('FAILED (conda read-only)')
        print('  [B] Downgrading pandas to 2.2.3...', end=' ')
        rB = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas==2.2.3'],
            capture_output=True, text=True
        )
        if rB.returncode == 0:
            print('OK')
            print('pandas downgraded successfully!')
            print('RESTART KERNEL NOW: Kernel -> Restart Kernel, then re-run from Cell 1')
        else:
            print(f'FAILED: {rB.stderr.strip()[:120]}')

In [ ]:
# ============================================================
# CELL 3 — Verify all packages (run AFTER kernel restart)
# ============================================================
print('=' * 60)
print('Verifying conda pre-installed packages...')
print('=' * 60)

import importlib

REQUIRED = [
    'torch', 'pandas', 'numpy', 'matplotlib',
    'transformers', 'datasets', 'evaluate',
    'peft', 'trl', 'bitsandbytes',
    'sacrebleu', 'accelerate', 'tokenizers',
    'rich', 'tqdm', 'scipy', 'sklearn',
]

all_ok = True
for mod in REQUIRED:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'installed')
        print(f'  OK  {mod:<20} {ver}')
    except ImportError as e:
        print(f'  XX  {mod:<20} NOT FOUND: {e}')
        all_ok = False

import numpy as np
import pandas as pd
print(f'\n  numpy  : {np.__version__}')
print(f'  pandas : {pd.__version__}')
major_np = int(np.__version__.split(".")[0])
major_pd = int(pd.__version__.split(".")[0])
if (major_pd >= 3 and major_np >= 2) or (major_pd < 3 and major_np < 2):
    print('  ABI match: compatible!')
else:
    print('  WARNING: ABI mismatch! Re-run Cell 2 and restart kernel.')

print('\nAll packages verified!' if all_ok else '\nSome packages missing!')

In [ ]:
# ============================================================
# CELL 4 — Verify GPU hardware
# ============================================================
import torch

print('=' * 60)
print('GPU Hardware Check')
print('=' * 60)
print(f'  PyTorch Version  : {torch.__version__}')
print(f'  CUDA Available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU Name         : {torch.cuda.get_device_name(0)}')
    print(f'  GPU VRAM         : {props.total_memory / 1e9:.1f} GB')
    print(f'  Compute Cap.     : {props.major}.{props.minor}')
    print(f'  CUDA Device      : cuda:{torch.cuda.current_device()}')
    print('\n  A100 Ready for QLoRA fine-tuning!')
else:
    print('\n  No GPU — training will run on CPU only.')

In [ ]:
# ============================================================
# CELL 5 — Verify Master Corpus & 0% Data Leakage
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
kineses_proj = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(kineses_proj):
    os.chdir(kineses_proj)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.integrity import DataLeakageChecker

print('=' * 60)
print('Master Corpus Verification')
print('=' * 60)

manager = MasterCorpusManager()
corpus  = manager.load_sentence_corpus()
lexical = manager.load_lexical_corpus()
train   = manager.load_train_split()
val     = manager.load_val_split()
test    = manager.load_test_split()

print(f'  Master Sentence Corpus : {len(corpus):,} multilingual concepts')
print(f'  Master Lexical Corpus  : {len(lexical):,} dictionary entries')
print(f'  Train Split            : {len(train):,} concepts (80%)')
print(f'  Val   Split            : {len(val):,} concepts (10%)')
print(f'  Test  Split            : {len(test):,} concepts (10%)')

print('\nRunning 0% leakage audit...')
checker = DataLeakageChecker(manager)
checker.verify_all()

print('\n' + '=' * 60)
print('  0% DATA LEAKAGE CONFIRMED')
print('  MASTER CORPUS LOADED')
print('  SETUP COMPLETE!')
print('=' * 60)
print('\nOpen any research notebook:')
print('  notebooks/05_instruction_generation.ipynb')
print('  notebooks/07_train_aya.ipynb')
print('  notebooks/08_train_llama.ipynb')
print('  notebooks/09_translation_evaluation.ipynb')